# Pediatric chest X-ray classification with ResNet-18

**Objective.** Compare a frozen-backbone baseline with full fine-tuning of ResNet-18 for binary classification (normal vs pneumonia). Choose the classification threshold on validation data to target high pneumonia recall, then measure performance on the original held-out test set.

**Data.** [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia). The images are pediatric radiographs from a limited clinical setting. This project does not establish clinical usefulness.

**Run setup.** Add the dataset under Kaggle **Input**, enable a GPU if available, and allow Internet for the pretrained weight download. Execute the notebook in order. Files are saved under `/kaggle/working/pneumonia_resnet18/`.

**Key limitation.** The available labels are used for an image-level train/validation split; patient-level independence is not verified. The original test folder is reserved for one final evaluation.

## 1. Environment and configuration

In [ ]:
import os, sys, random, json, subprocess, importlib.util
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    precision_score, recall_score, ConfusionMatrixDisplay
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| Device:", DEVICE)
if DEVICE.type != "cuda":
    print("WARNING: No GPU detected. Enable Kaggle Settings > Accelerator > GPU for speed.")

OUT_DIR = Path("/kaggle/working/pneumonia_resnet18") if Path("/kaggle/working").exists() else Path("./pneumonia_resnet18")
OUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH = OUT_DIR / "resnet18_best.pth"

BATCH_SIZE = 32
HEAD_EPOCHS = 3
FULL_EPOCHS = 6
TARGET_VAL_RECALL = 0.95
NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0
print("Output directory:", OUT_DIR.resolve())

## 2. Dataset inventory

In [ ]:
VALID_EXTENSIONS = {".jpeg", ".jpg", ".png"}

def locate_data_root():
    candidates = []
    for base in [Path("/kaggle/input"), Path("./data"), Path("/content")]:
        if not base.exists():
            continue
        for train_dir in base.rglob("train"):
            parent = train_dir.parent
            if (parent / "val").is_dir() and (parent / "test").is_dir():
                if all((train_dir / cls).is_dir() for cls in ["NORMAL", "PNEUMONIA"]):
                    candidates.append(parent)
    if not candidates:
        raise FileNotFoundError(
            "Dataset not found. Kaggle: Add Input > paultimothymooney/chest-xray-pneumonia; "
            "Colab: extract the dataset under ./data/chest_xray/"
        )
    # Both nested and flat versions occur in Kaggle dataset mounts.
    candidates = sorted(set(candidates), key=lambda p: (len(p.parts), str(p)))
    return candidates[0]

DATA_ROOT = locate_data_root()
CLASS_TO_IDX = {"NORMAL": 0, "PNEUMONIA": 1}
print("Data root:", DATA_ROOT)

def collect_samples(split_name):
    split_root = DATA_ROOT / split_name
    samples = []
    for label_name, label in CLASS_TO_IDX.items():
        folder = split_root / label_name
        if not folder.is_dir():
            raise FileNotFoundError(f"Missing class folder: {folder}")
        for path in sorted(folder.rglob("*")):
            if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS:
                samples.append((str(path), label))
    return samples

train_original = collect_samples("train")
val_original = collect_samples("val")
test_samples = collect_samples("test")  # Paths/labels only; DO NOT load test pixels yet.

def show_count(name, samples):
    counts = Counter(label for _, label in samples)
    print(f"{name:>16}: NORMAL={counts[0]:4d}, PNEUMONIA={counts[1]:4d}, total={len(samples):4d}")

show_count("Original train", train_original)
show_count("Original val", val_original)
show_count("Official test", test_samples)

## 3. Training and validation split

The original validation folder contains only 16 images. I combine the original training and validation folders, then generate a reproducible stratified 85/15 split. The official test set remains separate. Stratification balances labels, but does not resolve possible shared patients or near-duplicate images across partitions.

In [ ]:
pool_samples = train_original + val_original
pool_labels = [label for _, label in pool_samples]
train_samples, val_samples = train_test_split(
    pool_samples, test_size=0.15, random_state=SEED, stratify=pool_labels
)

assert set(path for path, _ in train_samples).isdisjoint(path for path, _ in val_samples)
assert set(path for path, _ in test_samples).isdisjoint(path for path, _ in pool_samples)
show_count("New train", train_samples)
show_count("New validation", val_samples)

## 4. Image transforms

Images are resized to 224 × 224, represented as three channels, and normalized using ImageNet statistics. Small affine and intensity changes are applied to training images only; evaluation images use fixed transforms.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomAffine(degrees=7, scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class XRayDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, index):
        path, label = self.samples[index]
        with Image.open(path) as im:
            image = ImageOps.exif_transpose(im).convert("RGB")
        return self.transform(image), label

train_ds = XRayDataset(train_samples, train_tfms)
val_ds = XRayDataset(val_samples, eval_tfms)
test_ds = XRayDataset(test_samples, eval_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
# Do not instantiate/iterate the test loader until the locked final evaluation cell.

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row, cls in enumerate([0, 1]):
    paths = [path for path, label in train_samples if label == cls][:4]
    for ax, path in zip(axes[row], paths):
        with Image.open(path) as im:
            ax.imshow(im.convert("L"), cmap="gray")
        ax.set_title("NORMAL" if cls == 0 else "PNEUMONIA")
        ax.axis("off")
plt.suptitle("Examples from training split only")
plt.tight_layout()
plt.show()

## 5. Model and training functions

I replace the ResNet-18 classification layer with two outputs (`NORMAL=0`, `PNEUMONIA=1`). Weighted cross-entropy increases the contribution of the less common normal class. Pneumonia recall is addressed separately when selecting the validation threshold.

In [ ]:
try:
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
except Exception as exc:
    raise RuntimeError(
        "Failed to download/load ImageNet pretrained weights. "
        "Turn on Kaggle Internet for the first run, or attach a pretrained weights file. "
        "Do not silently switch to random initialization."
    ) from exc

model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

counts = Counter(label for _, label in train_samples)
n_normal, n_pneumonia = counts[0], counts[1]
class_weights = torch.tensor([n_pneumonia / n_normal, 1.0], dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print("Class weight NORMAL:", round(class_weights[0].item(), 3), "| PNEUMONIA: 1.0")

@torch.inference_mode()
def predict_proba(model, loader):
    model.eval()
    labels, scores = [], []
    for images, targets in loader:
        logits = model(images.to(DEVICE, non_blocking=True))
        probabilities = logits.softmax(dim=1)[:, 1]
        labels.extend(targets.numpy().tolist())
        scores.extend(probabilities.cpu().numpy().tolist())
    return np.asarray(labels, dtype=np.int64), np.asarray(scores, dtype=np.float64)

def train_one_epoch(model, optimizer, head_only):
    model.train()
    if head_only:
        # Freezing parameters alone does NOT freeze BatchNorm running statistics.
        for layer in model.modules():
            if isinstance(layer, nn.BatchNorm2d):
                layer.eval()
    total_loss, total_n = 0.0, 0
    for images, targets in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(targets)
        total_n += len(targets)
    return total_loss / total_n

best_auc = -1.0
best_meta = {}
history = []

def run_phase(phase, epochs, lr, head_only):
    global best_auc, best_meta
    for parameter in model.parameters():
        parameter.requires_grad_(not head_only)
    if head_only:
        for parameter in model.fc.parameters():
            parameter.requires_grad_(True)
    optimizer = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad), lr=lr, weight_decay=1e-4
    )
    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(model, optimizer, head_only)
        y_val, p_val = predict_proba(model, val_loader)
        auc = roc_auc_score(y_val, p_val)
        r05 = recall_score(y_val, (p_val >= 0.5).astype(int), zero_division=0)
        row = {"phase": phase, "epoch": epoch, "loss": float(loss),
               "val_auc": float(auc), "val_recall_at_0.5": float(r05)}
        history.append(row)
        print(f"{phase:10s} epoch {epoch}/{epochs} | loss={loss:.4f} | "
              f"val AUC={auc:.4f} | val recall@0.5={r05:.3f}")
        if auc > best_auc:
            best_auc = float(auc)
            best_meta = {"phase": phase, "epoch": epoch, "val_auc": float(auc)}
            torch.save({"state_dict": model.state_dict(), "class_to_idx": CLASS_TO_IDX,
                        "seed": SEED, "best_validation": best_meta}, BEST_PATH)
            print("  Saved best validation-AUC checkpoint.")

print("Ready to train. Expected time varies with Kaggle GPU; no scores are known yet.")

## 6. Baseline: classification layer only

In [ ]:
run_phase("head_only", HEAD_EPOCHS, lr=1e-3, head_only=True)
baseline_auc = float(best_auc)
print("Best baseline validation ROC-AUC:", round(baseline_auc, 4))

## 7. Fine-tuning

All network parameters are unfrozen, with a lower learning rate than the head-only stage. Checkpoints are selected by validation ROC-AUC, with the baseline retained if fine-tuning does not improve it.

In [ ]:
# Start fine-tuning from the strongest head-only checkpoint.
baseline_ckpt = torch.load(BEST_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(baseline_ckpt["state_dict"])
run_phase("fine_tune", FULL_EPOCHS, lr=1e-4, head_only=False)

print("Best checkpoint:", best_meta)
print("Baseline AUC:", round(baseline_auc, 4), "| Best AUC:", round(best_auc, 4))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(history) + 1), [row["val_auc"] for row in history], marker="o")
ax.axvline(HEAD_EPOCHS + 0.5, color="gray", linestyle="--", label="Unfreeze backbone")
ax.set(xlabel="Epoch", ylabel="Validation ROC-AUC", title="Training history")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "validation_history.png", dpi=180)
plt.show()

## 8. Threshold selection

From the validation scores, select the largest decision threshold that reaches at least 95% recall for the pneumonia class. This is a validation target, not a guarantee for independent data. Precision is measured at the selected threshold.

In [ ]:
best_ckpt = torch.load(BEST_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(best_ckpt["state_dict"])
model.eval()

y_val, p_val = predict_proba(model, val_loader)
positive_scores = np.sort(p_val[y_val == 1])[::-1]
required_positive = int(np.ceil(TARGET_VAL_RECALL * len(positive_scores)))
THRESHOLD = float(positive_scores[required_positive - 1])
pred_val = (p_val >= THRESHOLD).astype(int)

val_recall = recall_score(y_val, pred_val, zero_division=0)
val_precision = precision_score(y_val, pred_val, zero_division=0)
print(f"Chosen threshold: {THRESHOLD:.6f}")
print(f"Validation pneumonia recall: {val_recall:.3%} (target {TARGET_VAL_RECALL:.0%})")
print(f"Validation pneumonia precision: {val_precision:.3%}")
print(f"Validation ROC-AUC: {roc_auc_score(y_val, p_val):.4f}")

best_ckpt["threshold"] = THRESHOLD
best_ckpt["target_val_recall"] = TARGET_VAL_RECALL
best_ckpt["validation_recall_at_threshold"] = float(val_recall)
best_ckpt["validation_precision_at_threshold"] = float(val_precision)
torch.save(best_ckpt, BEST_PATH)

with open(OUT_DIR / "training_history.json", "w", encoding="utf-8") as file:
    json.dump(history, file, indent=2)

## 9. Official holdout evaluation

The original test set is evaluated once after all choices are fixed. Record pneumonia recall, precision, ROC-AUC, and the full confusion matrix. Do not alter the model or threshold in response to these results and report another test score as if it were an untouched holdout.

In [ ]:
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
y_test, p_test = predict_proba(model, test_loader)
pred_test = (p_test >= THRESHOLD).astype(int)

test_auc = roc_auc_score(y_test, p_test)
test_recall = recall_score(y_test, pred_test, zero_division=0)
test_precision = precision_score(y_test, pred_test, zero_division=0)
cm = confusion_matrix(y_test, pred_test, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

metrics = {
    "model": "ResNet-18 (ImageNet transfer learning)",
    "seed": SEED,
    "dataset": "paultimothymooney/chest-xray-pneumonia",
    "split": "original train + val -> 85/15 stratified; official test held out",
    "image_level_split_patient_independence_unverified": True,
    "selection": best_meta,
    "target_validation_recall": TARGET_VAL_RECALL,
    "validation_recall": float(val_recall),
    "validation_precision": float(val_precision),
    "threshold": THRESHOLD,
    "test_samples": int(len(y_test)),
    "test_prevalence": float(np.mean(y_test)),
    "test_recall": float(test_recall),
    "test_precision": float(test_precision),
    "test_roc_auc": float(test_auc),
    "confusion_matrix_normal_first": cm.tolist(),
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
}

print("=== OFFICIAL TEST RESULTS (ONE-TIME) ===")
print(f"Recall (pneumonia): {test_recall:.3%} | Precision: {test_precision:.3%} | ROC-AUC: {test_auc:.4f}")
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}  |  Threshold={THRESHOLD:.6f}")
if test_recall < TARGET_VAL_RECALL:
    print("NOTE: Test recall fell below the validation target. Report this honestly; do not retune on test.")

with open(OUT_DIR / "metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["NORMAL", "PNEUMONIA"]).plot(
    ax=axes[0], cmap="Blues", colorbar=False, values_format="d"
)
fpr, tpr, _ = roc_curve(y_test, p_test)
axes[1].plot(fpr, tpr, label=f"Test ROC-AUC = {test_auc:.3f}")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[1].set(xlabel="False positive rate", ylabel="True positive rate", title="Official test ROC curve")
axes[1].legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "official_test_evaluation.png", dpi=180)
plt.show()

## 10. Grad-CAM inspection

For representative true positives, true negatives and errors, generate a Grad-CAM overlay for the pneumonia score using the last residual block. These are approximate attribution maps, not lesion annotations. Activation near markers or image borders would warrant further investigation.

In [ ]:
if importlib.util.find_spec("pytorch_grad_cam") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "grad-cam"])

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def overlay_pneumonia_cam(pil_image):
    rgb = ImageOps.exif_transpose(pil_image).convert("RGB").resize((224, 224))
    image_array = np.asarray(rgb, dtype=np.float32) / 255.0
    input_tensor = eval_tfms(rgb).unsqueeze(0).to(DEVICE)
    # Grad-CAM needs gradients: DO NOT wrap this call in no_grad()/inference_mode().
    with torch.enable_grad():
        with GradCAM(model=model, target_layers=[model.layer4[-1]]) as cam:
            cam_mask = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(1)])[0]
    return Image.fromarray(show_cam_on_image(image_array, cam_mask, use_rgb=True))

cases = {
    "TP": np.where((y_test == 1) & (pred_test == 1))[0],
    "TN": np.where((y_test == 0) & (pred_test == 0))[0],
    "FP": np.where((y_test == 0) & (pred_test == 1))[0],
    "FN": np.where((y_test == 1) & (pred_test == 0))[0],
}
available = [(name, int(ids[0])) for name, ids in cases.items() if len(ids)]
if available:
    fig, axes = plt.subplots(len(available), 2, figsize=(7, 3.6 * len(available)), squeeze=False)
    for row, (case, index) in enumerate(available):
        with Image.open(test_samples[index][0]) as im:
            original = im.convert("RGB")
        overlay = overlay_pneumonia_cam(original)
        axes[row, 0].imshow(original, cmap="gray")
        axes[row, 0].set_title(f"{case}: original | pneumonia score={p_test[index]:.3f}")
        axes[row, 1].imshow(overlay)
        axes[row, 1].set_title("Grad-CAM for PNEUMONIA logit")
        for ax in axes[row]:
            ax.axis("off")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "gradcam_examples.png", dpi=180)
    plt.show()
else:
    print("No examples available for visualization.")

## 11. Optional Gradio interface

An upload interface displays the model flag, its uncalibrated pneumonia score and the Grad-CAM overlay. This is a local demonstration only; it should not be used for diagnosis or on private patient data.

In [ ]:
if importlib.util.find_spec("gradio") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio"])

import gradio as gr

def demo_predict(pil_image):
    if pil_image is None:
        return "Please upload an image.", None, None
    img = ImageOps.exif_transpose(pil_image).convert("RGB")
    model.eval()
    with torch.inference_mode():
        logits = model(eval_tfms(img).unsqueeze(0).to(DEVICE))
        pneumonia_score = float(logits.softmax(dim=1)[0, 1].item())
    flag = "MODEL FLAG: PNEUMONIA" if pneumonia_score >= THRESHOLD else "MODEL FLAG: NORMAL"
    return flag, round(pneumonia_score, 4), overlay_pneumonia_cam(img)

demo = gr.Interface(
    fn=demo_predict,
    inputs=gr.Image(type="pil", label="Upload a chest X-ray"),
    outputs=[gr.Textbox(label="Experimental model flag"),
             gr.Number(label="Pneumonia model score (not calibrated)"),
             gr.Image(type="pil", label="Grad-CAM (not lesion localization)")],
    title="Pediatric X-ray pneumonia classification — research demo",
    description=(
        "Educational project only — NOT a medical device or diagnostic service. "
        "Never use for medical decisions. The model was trained on a limited pediatric dataset; "
        "predictions and attention maps can be incorrect."
    ),
    flagging_mode="never",
)
demo.launch(share=False, prevent_thread_lock=True)

## 12. Reporting and limitations

Publish the values from `metrics.json`, the validation history, confusion matrix, ROC plot, and representative Grad-CAM examples. Distinguish baseline and fine-tuning results. Relevant limitations include uncertain patient independence, a single pediatric data source, potential acquisition shortcuts, and lack of external validation. Softmax scores should not be interpreted as calibrated disease probabilities.

**Methods references:** [TorchVision ResNet-18](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.resnet18), [Grad-CAM implementation](https://github.com/jacobgil/pytorch-grad-cam), [Grad-CAM paper](https://arxiv.org/abs/1610.02391).